In [5]:
import supervision as sv
import os
import shutil

In [11]:
import json
import supervision as sv
import os

# Create directory structure
base_dirs = {
    'coco': 'rfdetr_dataset',
    'yolo': 'yolo_dataset'
}

# Step 1: Load original COCO JSON
with open("Dataset/_annotations.coco.json", "r") as f:
    coco_data = json.load(f)

# Step 2: Remap category IDs from 1–11 to 0–10
for ann in coco_data["annotations"]:
    ann["category_id"] -= 1

for cat in coco_data["categories"]:
    cat["id"] -= 1

# Step 3: Save modified JSON temporarily
adjusted_json_path = "Dataset/_annotations.coco.adjusted.json"
with open(adjusted_json_path, "w") as f:
    json.dump(coco_data, f)

# Step 4: Load adjusted dataset
ds = sv.DetectionDataset.from_coco(
    images_directory_path="Dataset/image",
    annotations_path=adjusted_json_path
)

# Step 5: Split dataset
ds_train, ds_temp = ds.split(split_ratio=0.7, shuffle=True, random_state=42)
ds_valid, ds_test = ds_temp.split(split_ratio=0.5, shuffle=True, random_state=42)

splits = {
    'train': ds_train,
    'valid': ds_valid,
    'test': ds_test
}

# Step 6: Save splits in both formats
first_split = True
for split_name, split_data in splits.items():
    # Save COCO format (original category IDs already adjusted)
    split_data.as_coco(
        images_directory_path=f"{base_dirs['coco']}/{split_name}",
        annotations_path=f"{base_dirs['coco']}/{split_name}/_annotations.coco.json"
    )

    # Save YOLO format
    split_data.as_yolo(
        images_directory_path=f"{base_dirs['yolo']}/{split_name}/images",
        annotations_directory_path=f"{base_dirs['yolo']}/{split_name}/labels",
        data_yaml_path=f"{base_dirs['yolo']}/data.yaml" if first_split else None
    )
    first_split = False

# Step 7: Update YOLO data.yaml with correct paths and class info
with open('Dataset/data.yaml', 'r') as f:
    original_yaml = f.read()

with open(f"{base_dirs['yolo']}/data.yaml", 'w') as f:
    f.write(f"""path: ./  # dataset root directory
train: train/images      # train images relative to path
val: valid/images        # validation images relative to path
test: test/images        # test images relative to path

{original_yaml[original_yaml.find('nc:'):]}  # preserve original class information
""")

In [13]:
import os

def validate_labels(label_dir, max_class_id=11):
    for fname in os.listdir(label_dir):
        with open(os.path.join(label_dir, fname), 'r') as f:
            for line in f:
                class_id = int(line.strip().split()[0])
                if class_id < 0 or class_id > max_class_id:
                    print(f"❌ Invalid class ID {class_id} in {fname}")

# Check all splits
for split in ['train', 'valid', 'test']:
    validate_labels(f"C:/Users/kaiva/Desktop/GitHub/Udacity Self Driving Car/udacity-self-driving-car/yolo_dataset/{split}/labels")